In [ ]:
import requests
import json

response = requests.get("http://localhost:8188/object_info")
node_info = response.json()

# Check KSampler inputs
print("KSampler inputs:", node_info["KSampler"]["input"]["required"].keys())

# Check VHS_VideoCombine inputs
print("VHS_VideoCombine inputs:", node_info["VHS_VideoCombine"]["input"]["required"].keys())

# Check ADE_AnimateDiffLoaderGen1 inputs
print("AnimateDiff inputs:", node_info["ADE_AnimateDiffLoaderGen1"]["input"]["required"].keys())

KSampler inputs: dict_keys(['model', 'seed', 'steps', 'cfg', 'sampler_name', 'scheduler', 'positive', 'negative', 'latent_image', 'denoise'])
VHS_VideoCombine inputs: dict_keys(['images', 'frame_rate', 'loop_count', 'filename_prefix', 'format', 'pingpong', 'save_output'])
AnimateDiff inputs: dict_keys(['model', 'model_name', 'beta_schedule'])


In [ ]:
minimal_workflow = {
    "1": {
        "class_type": "CheckpointLoaderSimple",
        "inputs": {
            "ckpt_name": "dreamshaper_8.safetensors"
        }
    },
    "2": {
        "class_type": "CLIPTextEncode",
        "inputs": {
            "text": "test prompt",
            "clip": ["1", 1]
        }
    },
    "3": {
        "class_type": "EmptyLatentImage",
        "inputs": {
            "batch_size": 1,
            "width": 512,
            "height": 512
        }
    },
    "4": {
        "class_type": "KSampler",
        "inputs": {
            "seed": 42,
            "steps": 20,
            "cfg": 8.0,
            "sampler_name": "euler",
            "scheduler": "normal",
            "denoise": 1.0,
            "model": ["1", 0],
            "positive": ["2", 0],
            "negative": ["2", 0],
            "latent_image": ["3", 0]
        }
    },
    "5": {
        "class_type": "VAEDecode",
        "inputs": {
            "samples": ["4", 0],
            "vae": ["1", 2]
        }
    },
    "6": {
        "class_type": "SaveImage",
        "inputs": {
            "filename_prefix": "test",
            "images": ["5", 0]
        }
    }
}

In [ ]:
def validate_workflow(workflow):
    """Basic validation of workflow structure"""
    issues = []
    
    for node_id, node in workflow.items():
        # Check if node has required fields
        if "class_type" not in node:
            issues.append(f"Node {node_id} missing class_type")
        if "inputs" not in node:
            issues.append(f"Node {node_id} missing inputs")
            
        # Check connections format
        for input_name, input_value in node["inputs"].items():
            if isinstance(input_value, list):
                if len(input_value) != 2:
                    issues.append(f"Node {node_id}, input {input_name}: connection should be [node_id, output_index]")
                else:
                    # Check if referenced node exists
                    if input_value[0] not in workflow:
                        issues.append(f"Node {node_id} references missing node {input_value[0]}")
    
    if issues:
        print("Validation issues found:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("✅ Basic validation passed")
    
    return len(issues) == 0

# Validate your workflow
validate_workflow(minimal_workflow)

✅ Basic validation passed


True

In [ ]:
import requests
import time
import random

# === CONFIGURATION ===
COMFYUI_URL = "http://localhost:8188"
CHECKPOINT = "dreamshaper_8.safetensors"  # Change to your checkpoint
MOTION_MODULE = "mm_sd_v15_v2.ckpt"       # Change to your motion module

# === NEGATIVE PROMPT ===
NEGATIVE_PROMPT = "nsfw, blurry, low quality, deformed, extra limbs, ugly, text, watermark"

# === YOUR PROMPT ===
YOUR_PROMPT = "anime girl smiling, cherry blossoms, soft lighting, high quality"

# === CREATE WORKFLOW ===
def create_workflow(prompt, seed=None):
    if seed is None:
        seed = random.randint(0, 2**31)
    
    return {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": CHECKPOINT}},
        "2": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["1", 1]}},
        "3": {"class_type": "CLIPTextEncode", "inputs": {"text": NEGATIVE_PROMPT, "clip": ["1", 1]}},
        "4": {"class_type": "EmptyLatentImage", "inputs": {"batch_size": 8, "width": 384, "height": 384}},
        "5": {"class_type": "ADE_AnimateDiffLoaderWithContext", "inputs": {
            "model": ["1", 0], "model_name": MOTION_MODULE, "beta_schedule": "autoselect",
            "context_options": None, "motion_lora": None, "ad_settings": None,
            "sample_settings": None, "motion_scale": 1.0, "apply_v2_models_properly": True
        }},
        "6": {"class_type": "KSampler", "inputs": {
            "seed": seed, "steps": 10, "cfg": 7.0, "sampler_name": "euler", "scheduler": "normal",
            "denoise": 1.0, "model": ["5", 0], "positive": ["2", 0], "negative": ["3", 0],
            "latent_image": ["4", 0]
        }},
        "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["1", 2]}},
        "8": {"class_type": "VHS_VideoCombine", "inputs": {
            "images": ["7", 0], "frame_rate": 8, "loop_count": 0,
            "filename_prefix": "test", "format": "video/h264-mp4",
            "pingpong": False, "save_output": True, "audio": None
        }}
    }

# === SEND TO COMFYUI ===
print("Sending workflow to ComfyUI...")
response = requests.post(f"{COMFYUI_URL}/prompt", json={"prompt": create_workflow(YOUR_PROMPT)})

if response.status_code != 200:
    print(f"Error: {response.text}")
    exit()

prompt_id = response.json()['prompt_id']
print(f"Job ID: {prompt_id}")

# === WAIT FOR COMPLETION ===
print("Generating (this will take 1-2 minutes for 8 frames)...")
start = time.time()

while True:
    # Check if done
    history = requests.get(f"{COMFYUI_URL}/history").json()
    if prompt_id in history:
        elapsed = time.time() - start
        print(f"\nDone! Took {elapsed:.1f} seconds")
        print("Check your ComfyUI/output folder for test.mp4")
        break
    
    # Check progress
    queue = requests.get(f"{COMFYUI_URL}/queue").json()
    running = len(queue.get('running', []))
    pending = len(queue.get('pending', []))
    
    elapsed = time.time() - start
    print(f"\r⏳ {elapsed:.0f}s | Running: {running} | Pending: {pending}", end="")
    
    time.sleep(2)